### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from hydra import compose, initialize
from pathlib import Path
from omegaconf import OmegaConf

initialize(config_path="../config", version_base="1.3")
cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))

sys.path.insert(0, str(cfg.paths.project_root))

paths:
  project_root: /home/p84400019/projects/consciousness-llms/IT-LLMs/
  model_path: ${model.company}/${model.model_family}/${model.model_size}/${model.it}/
  activation_method: ${time_series.node_type}/${time_series.node_activation}/${time_series.projection_method}/
  phyid_method: ${paths.activation_method}phyid_tau-${phyid.tau}/phyid_kind-${phyid.kind}/phyid_redundancy-${phyid.redundancy}/
  data_dir: ${paths.project_root}data/${paths.model_path}
  data_activations_dir: ${paths.data_dir}activations/
  data_activations_file: ${paths.data_activations_dir}multi_prompt_activations.pkl
  data_phyid_dir: ${paths.data_dir}phyid/${paths.phyid_method}
  data_phyid_file: ${paths.data_phyid_dir}multi_prompt_phyid.pkl
  plot_dir: ${paths.project_root}plots/${paths.model_path}
  plot_time_series_dir: ${paths.plot_dir}time_series/${paths.activation_method}
  plot_phyid_dir: ${paths.plot_dir}phyid/${paths.phyid_method}
model:
  shortcode: D2-16-A2
  hf_name: deepseek-ai/DeepSeek-V2-Lite
  com

### Time Series Loading

In [2]:
from src.activation_recorder import MultiPromptActivations

data_activations_file = cfg.paths.data_activations_file
activations = MultiPromptActivations.load(file_path=data_activations_file)

/home/p84400019/miniconda3/envs/int/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MultiPromptActivations successfully loaded from '/home/p84400019/projects/consciousness-llms/IT-LLMs/data/deepseek/deepseek-v2/16B-A2B/base/activations/multi_prompt_activations.pkl'.


In [3]:
from src.time_series_activations import MultiPromptTimeSeries

time_series = MultiPromptTimeSeries.from_activations(
    activations, 
    node_type=cfg.time_series.node_type,
    node_activation=cfg.time_series.node_activation,
    projection_method=cfg.time_series.projection_method, 
    exclude_shared_expert_moe=cfg.time_series.exclude_shared_expert_moe, 
)
time_series.plot(token_x=True, ticks_all_layers=True, plot_dir=cfg.paths.plot_time_series_dir)

Prompt 0 has 128 generated tokens: ['\n', 'Imagine', ' a', ' future', ' where', ' humans', ' have', ' evolved', ' to', ' live', ' underwater', '.', ' Describe', ' the', ' adaptations', ' they', ' might', ' develop', '.', '\n', 'The', ' human', ' body', ' is', ' a', ' complex', ' machine', ' that', ' is', ' constantly', ' adapting', ' to', ' the', ' environment', ' in', ' which', ' it', ' lives', '.', ' In', ' a', ' future', ' where', ' humans', ' have', ' evolved', ' to', ' live', ' underwater', ',', ' the', ' body', ' would', ' need', ' to', ' adapt', ' to', ' the', ' new', ' environment', ' in', ' order', ' to', ' survive', '.', '\n', 'One', ' of', ' the', ' most', ' important', ' adaptations', ' would', ' be', ' the', ' lungs', '.', ' In', ' a', ' future', ' where', ' humans', ' live', ' underwater', ',', ' the', ' lungs', ' would', ' need', ' to', ' be', ' adapted', ' to', ' allow', ' for', ' the', ' absorption', ' of', ' oxygen', ' from', ' the', ' water', '.', ' This', ' would', 

### PhyID Decomposition

In [4]:
from src.phyid_decomposition import MultiPromptPhyID, PromptPhyID, PhyIDTimeSeries
compute_phyid = True
data_phyid_file = cfg.paths.data_phyid_file

if compute_phyid:
    phyid_comp = MultiPromptPhyID.from_time_series(
        time_series,
        cfg.phyid.tau,
        cfg.phyid.kind,
        cfg.phyid.redundancy
    ) 
    phyid_comp.save(file_path=data_phyid_file)

Processing prompt 1/1 with 128 generated tokens.
[ETA] 1/186192 done | avg=0.029s | ETA ≈ 1:31:28
[ETA] 10/186192 done | avg=0.005s | ETA ≈ 0:15:11
[ETA] 100/186192 done | avg=0.003s | ETA ≈ 0:10:08
[ETA] 1000/186192 done | avg=0.002s | ETA ≈ 0:07:34
[ETA] 10000/186192 done | avg=0.002s | ETA ≈ 0:06:27
[ETA] 100000/186192 done | avg=0.002s | ETA ≈ 0:03:21
[ETA] 186192/186192 done | avg=0.002s | ETA ≈ 0:00:00
MultiPromptPhyID successfully saved to '/home/p84400019/projects/consciousness-llms/IT-LLMs/data/deepseek/deepseek-v2/16B-A2B/base/phyid/attention/attention_outputs/max/phyid_tau-1/phyid_kind-gaussian/phyid_redundancy-MMI/multi_prompt_phyid.pkl'.


In [5]:
phyid = MultiPromptPhyID.load(file_path=data_phyid_file)
phyid.compute_extra_atoms()
phyid = phyid.get_prompt(prompt_index=0)  # Get the first prompt's phyid
phyid.build_data_array()

MultiPromptPhyID successfully loaded from '/home/p84400019/projects/consciousness-llms/IT-LLMs/data/deepseek/deepseek-v2/16B-A2B/base/phyid/attention/attention_outputs/max/phyid_tau-1/phyid_kind-gaussian/phyid_redundancy-MMI/multi_prompt_phyid.pkl'.


/home/p84400019/projects/consciousness-llms/IT-LLMs/src/phyid_decomposition/MultiPromptPhyID.py:140: RuntimeWarning: divide by zero encountered in divide
  setattr(self, f"{atom}_normalized", getattr(self, atom) / mi)
/home/p84400019/projects/consciousness-llms/IT-LLMs/src/phyid_decomposition/MultiPromptPhyID.py:140: RuntimeWarning: invalid value encountered in divide
  setattr(self, f"{atom}_normalized", getattr(self, atom) / mi)


<xarray.DataArray 'phiid' (atom: 54, source_layer: 27, source_node: 16,
                           target_layer: 27, target_node: 16, time: 127)> Size: 5GB
array([[[[[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
             0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
           [ 7.65603781e-02, -4.66401689e-02, -4.36649919e-01, ...,
            -1.42536126e-02, -3.67524289e-02, -7.71486908e-02],
           [ 1.12719856e-01, -1.54007852e-01,  9.92818922e-02, ...,
             1.28253624e-01,  2.79900432e-02,  2.18370184e-02],
           ...,
           [ 3.78127187e-01, -2.30236918e-01, -3.07375968e-01, ...,
             2.34355807e-01, -1.44036591e-01, -1.58798218e-01],
           [ 1.44431040e-01, -1.96212791e-02, -5.47924526e-02, ...,
             2.17348278e-01, -5.16407117e-02, -6.07317314e-02],
           [ 8.93736910e-03,  1.57212213e-01,  2.54819185e-01, ...,
            -1.36185527e-01,  8.15092400e-03,  1.15130059e-02]],

          [[ 1.99090183e-01, -1.40965462e-01,  1.87540576e-02, ...,
             1.26600063e+00, -5.54730415e-01, -2.73276091e-01],
           [ 1.58768326e-01, -1.34516239e-01,  2.62057394e-01, ...,
             1.79952569e-02, -2.89784875e-02, -6.71480000e-02],
           [ 1.14813495e+00,  1.58852935e-01,  6.58089817e-02, ...,
             1.30819988e+00, -8.84757400e-01, -6.66171193e-01],
...
             2.80793786e+00,  3.88304234e-01,  4.46888804e-01],
           [-5.67482971e-02,  1.65201461e+00, -9.01587084e-02, ...,
            -4.73498011e+00, -8.88661087e-01,  1.30319715e-01],
           [-2.06592560e+00, -3.79403725e+01,  8.75990391e-01, ...,
            -2.29297412e+03,  3.55675250e-01, -2.35805655e+00]],

          [[-3.92587280e+01, -6.17389011e+00,  7.84296930e-01, ...,
            -8.77399266e-01,  5.68529725e-01,  5.53049240e+01],
           [ 1.48789811e+00,  6.25794649e+00,  7.98617005e-01, ...,
            -2.91828424e-01,  2.08993149e+00, -2.75918484e-01],
           [ 2.55567193e+00,  1.41236615e+00,  9.21540022e-01, ...,
             5.71909428e-01,  6.01740456e+01,  2.18446779e+00],
           ...,
           [-8.74105632e-01,  4.47597647e+00,  7.15116024e-01, ...,
             6.61579609e-01, -4.87506300e-01, -4.24977779e-01],
           [ 7.10163638e-02,  3.02567631e-01,  1.08480060e+00, ...,
             1.77369833e+00,  5.98476791e+00, -2.61535692e+00],
           [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
             0.00000000e+00,  0.00000000e+00,  0.00000000e+00]]]]]],
      dtype=float32)
Coordinates:
  * atom          (atom) <U56 12kB 'causal_density' ... 'yty_normalized'
  * source_layer  (source_layer) int64 216B 0 1 2 3 4 5 6 ... 21 22 23 24 25 26
  * source_node   (source_node) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * target_layer  (target_layer) int64 216B 0 1 2 3 4 5 6 ... 21 22 23 24 25 26
  * target_node   (target_node) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * time          (time) int64 1kB 0 1 2 3 4 5 6 ... 120 121 122 123 124 125 126
Attributes:
    model:    deepseek-ai/DeepSeek-V2-Lite

In [6]:
plot_dir = cfg.paths.plot_phyid_dir
# plot_dir = None
phyid.node_heatmap(atom='sts', plot_dir=plot_dir)
phyid.plot_mean_along('sts', varying_dim='source_layer', plot_dir=plot_dir)
phyid.plot_mean_along('sts_normalized', varying_dim='source_layer', plot_dir=plot_dir)
phyid.plot_mean_along('mutual_information', varying_dim='source_layer', plot_dir=plot_dir)

Heatmap saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/attention/attention_outputs/max/phyid_tau-1/phyid_kind-gaussian/phyid_redundancy-MMI/node_heatmap/sts/heatmap.png
Plot saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/attention/attention_outputs/max/phyid_tau-1/phyid_kind-gaussian/phyid_redundancy-MMI/plot_mean_along/sts/source_layer.png
Plot saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/attention/attention_outputs/max/phyid_tau-1/phyid_kind-gaussian/phyid_redundancy-MMI/plot_mean_along/sts_normalized/source_layer.png
Plot saved to /home/p84400019/projects/consciousness-llms/IT-LLMs/plots/deepseek/deepseek-v2/16B-A2B/base/phyid/attention/attention_outputs/max/phyid_tau-1/phyid_kind-gaussian/phyid_redundancy-MMI/plot_mean_along/mutual_information/source_layer.png
